# Trabajo final · Data Analyst · EduNova

**Plataforma de cursos online** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real de ARCA.

## La pregunta oficial

> **¿Qué cursos funcionan mejor? (ranking por tasa de finalización)**

**Decisión que habilita:** En qué cursos invertir en actualización y promoción.

Antes de analizar, **verificá si los datos sostienen la premisa de la pregunta**. A veces no la sostienen, y decirlo es un hallazgo.

Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Analyst · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 2 | SQL y PL/SQL |
| 3 | Estadística |
| 4 | Python: limpieza |
| 5 | Proceso de análisis |
| 6 | Superset |
| 7 | Analítica de negocio |
| 8 | Bitácora y entrega |

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## 1 · La base de EduNova

La primera celda **genera la base** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda la carga en SQLite y te deja tres herramientas: `sql(...)` para consultar, `esquema()` para ver las tablas y `anotar(...)` para dejar registrado cada problema de calidad que encuentres.

In [ ]:
import zlib, base64
_BLOB = (
    "eNp1Vstu3DYU3fsrCAEGqESjSPNwJnZnESRtkUW9SheFIAxoiTNmLZEyJU3tGP6YLrvoIugn+Md6Lql5OfbAlijeB+8990VV"
    "N8Z2TPd1c89Ey3QTsUboEkv8NeVJKVds2faFMtxGrJVWyYhthFVCd7IFc3h+wvBr2cJT48I09zyMRdvdN5Kbqz9l0YUXrAaD"
    "jS10m5pXUvM2DNlPrPHSWZ2Dnu0U803EbMhWxrINU9ox5BfMyq63+DrxdtXivm+Xsm0E2edEzrc82SauzF/S8hD2xn3TDEv2"
    "lgUsiOiB5SbPbKxw4lralicRm4a5V16K6VKWvTYbwVspy8U0mQ6+Wpiqm8GXGMyir7ql1WvHCF+FlYIAyYLPojMtnfabsDey"
    "U3pNH19KZWrh9j+rVj59N7T8VbadevpPB3BUq42spNfxRatCicoJwlJbS4jT18eN0N9EaYLcmVX0tjUk0pQxzhW/WFFL/hCo"
    "culIwTnLVsGnB3WejMvHwIGrCFw4spY8jdgkDXPoJfOJ2bmRKXbKZvlz7kniWJ2dxDsYnHHF3r1jsxBCk9eEGisRL0jZuLg2"
    "qpA8G39IEqA/8a+pf53RCwIQg1TZWwqzXl4fSc4jlo4jNibpgfkRIbh16aaNrUVFcY2nM0e7YE3tw1dUquHYZiMWz9gbSLw9"
    "lkjnw9FxAtn4QxIewJwFy9uAcvb2gl3iNYOp0C0QTh+1j3YtNQIuKFKfrlUlafG77de9uHcZ8fTPnSpcID+ZytRXyrH+jGx+"
    "+i4oCUTV1/qViHqaD+lHCunshZBeerhh1CFk3siIXaJ6F5kDJk7pkcxpNaXHmD7PcsJ9JYtrsbRyrdrO0okc1nxVNbJV1A0P"
    "xsl4OkrORkkahIAQxM4sO9BLWXWCH5fXbIrHJdT2WnWL4HMQhjHUroifB6d/jE7r0WkZ0LnGKiB4FGvsrZ/+1gNqonQVZOUK"
    "faf0W5WikgjynXcTcmwy37qYjnOkh4vjSlW+SIH0Dd6J3wWCxVHG+pqnn1ZggzeHPo1ncOj9LAmp7CuXdIf+wgqtwp0G0i5I"
    "u6j2ah2BkjKokTHGPv0rAqZW+27p+mSdFTmTVStZsLbiisr+gn2DFB9REoOfdCz2VM+cxhOKyi1JP8vvGUwuTN0cNmY6KmXv"
    "GE/BjSKRdw0ffSPnAD0Yh5SMfXAy4XqyrJ+r4ACaTHJCCx8qb0+cTMIjz5FG5LooOrV5wW9koRfkAR10ZapWlAMfHTzQyI9K"
    "dluSc8uDBTCgTBOF3LhhbxcsPbKgsYZcezhUQlmHFEW8av4BYKWJ60EHyojjCsw8jcc0NtBBzhKXjnsr9zxoZsRB2b919fAI"
    "ys35LHzMgAYg9Q0SNq0qIzruO07sdxFI0oRjzw6CfpA5PuzE46X35ZNSF8P/PPe18R5rKvQ0D49j4kojFpiZuuR8FXx5uDlP"
    "ztBgIkbdRvhuEw2tMN4OGJgWvdAdZmgNB93h6641lOK+XTyrJ2rjKLrw1b6wqjHmTQ/DPB4Ro8jguP22WSNgxOr837Z6mtiO"
    "YVcE7+MpTAKOb1x9QCiehl40pZDjggKEXYIC4sMUcxhDoRZ6gE7pH/q0QxEgGaqZdpFR2wZbYVVDc4z61b6R+w8/p7dN1y3I"
    "1M7teX+XjVgL3+0I43IgmbWVLYhFR9/adGK5wvCpghxJvzOuMLoAJBk2ItqNW0FO8Tlg90W3hM5OLtxFBoioNbCSMLuUd4uv"
    "tpfhVmSFabxIXxKLYYnsvAwvrWm84G6eZX4i0fDc3yy3jYVIEatEfVUKRjc6e/7qVS47H+c/3OxwofvhRudmm49TjQji4GGI"
    "Z0eg00RH9e3uUUN6kwt8G0ViCncXUfQMx4Ni9gs3g5wn2BpWFNp92CVRHP7BkozBF70eT07QUKUVdLk8uHue/A8x1oER"
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [ ]:
import sqlite3, matplotlib.pyplot as plt
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); tablas = {k: v for k, v in t.items() if not k.startswith("_")}
con = sqlite3.connect(":memory:")
for nombre, df in tablas.items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False)

def sql(consulta, parametros=None):
    """Corre una consulta SQL sobre la base y devuelve un DataFrame."""
    return pd.read_sql_query(consulta, con, params=parametros)

def esquema():
    """Lista las tablas, sus columnas y sus filas."""
    for nombre, df in tablas.items(): print(f"{nombre}  ({len(df):,} filas)".replace(",", "."), "\n   ", ", ".join(df.columns))

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset"):
    """Guarda la base como archivo SQLite y un CSV por tabla, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); salida = sqlite3.connect(f"{carpeta}/base.db")
    for nombre, df in tablas.items(): df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, salida, index=False, if_exists="replace")
    salida.close(); print(f"Listo: {carpeta}/base.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta
import os
esquema()

Los datos tienen los problemas de un dato real. Encontrarlos, decidir qué hacer con cada uno y dejarlo escrito es parte de la evidencia.

## 2 · SQL y PL/SQL

Tres consultas. Guardalas en esta sección: son la evidencia del módulo.

1. La finalización por curso (JOIN de `inscripciones` con `cursos`).
2. La finalización por curso y por formato.
3. Los reembolsos por origen del alumno (JOIN con `alumnos`).

SQLite no tiene PL/SQL. Si querés encapsular una lógica, escribila como una función de Python que ejecuta SQL.

In [ ]:
sql("""
-- Consulta 1
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 2
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 3 (control de integridad)
SELECT 1
""")

## 3 · Estadística

Para la variable principal (`progreso_pct`): **media, mediana, dispersión y distribución**. Decidí cuál resume mejor y escribí por qué.

In [ ]:
# Tu análisis estadístico

## 4 · Python: limpieza

Traé las tablas a pandas (`df = sql("SELECT * FROM tabla")`) y documentá la calidad: **faltantes, duplicados, valores imposibles y categorías inconsistentes**. Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)`.

In [ ]:
# Tu limpieza
# anotar("problema", "tabla.columna", 0, "qué decidimos y por qué")

In [ ]:
bitacora_calidad

## 5 · Proceso de análisis

**Verificá la premisa** de la pregunta oficial antes de seguir. Después armá al menos dos variables nuevas (formato, origen del alumno, mes de inscripción, descuento aplicado).

In [ ]:
# Tu verificación de la premisa y tus variables nuevas

## 6 · Dashboard en Superset

Exportá la base y conectá Apache Superset al archivo. Armá **4 a 6 gráficos, uno por pregunta**, con un orden de lectura y las definiciones a la vista. Si no podés usar Superset, trabajá con gráficos de este notebook.

In [ ]:
exportar_superset()

In [ ]:
# Si no usás Superset, tus gráficos con matplotlib

## 7 · Analítica de negocio

**KPIs** con su denominador: tasa de finalización (completadas sobre completadas más abandonadas), reembolsos, nota final, ingreso por inscripción.

**Segmentación:** curso, formato, origen del alumno y área.

Un **insight** relevante, una **limitación** y una **recomendación** proporcional a la evidencia.

In [ ]:
# Tus KPIs y tu segmentación

## 8 · Bitácora y entrega

Completá esta bitácora editando la celda. Con ella, otra persona tiene que poder rehacer tus números.

**Equipo:** ____  **Integrantes y roles:** ____

**Pregunta y premisa:** ¿los datos sostienen la premisa? ____

| KPI | Fórmula | Universo | Período | Estados o faltantes |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Hechos** (lo que muestran los datos): ____

**Inferencias** (lo que creemos que significa): ____

**Lo que contradice nuestra hipótesis:** ____

**Recomendación** (proporcional a la evidencia): ____

**Límites y siguiente análisis:** ____

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Mi pregunta termina en una decisión y nombra quién la toma.
- [ ] Verifiqué la premisa de la pregunta.
- [ ] Cada KPI declara fórmula, universo, período y tratamiento de estados.
- [ ] Ningún porcentaje se muestra sin decir sobre cuántos casos.
- [ ] La limpieza está anotada: qué encontré, qué hice y cuántas filas afectó.
- [ ] Los hechos están separados de las inferencias y no prometo causalidad.
- [ ] Mostré al menos un resultado que no me conviene.
- [ ] Hay una limitación y un siguiente análisis.